# Trie-native ARC-AGI-2 code submission
This Internet-disabled notebook uses a hash-pinned static native worker and writes `/kaggle/working/submission.json`. It performs no package installation, compilation, upload, or Kaggle API operation.
Every construction is induced from all training demonstrations and must replay them exactly before query execution. Native abstentions and failures retain their status and receive a separately labelled copy-input format fallback.
Each output has two attempts; the second uses at most one separately represented, all-demonstration-replayed construction when applicable and output-distinct, otherwise it honestly duplicates the first. Full prediction/status streams freeze before any optional separate solution file is loaded.


In [ ]:
from pathlib import Path
import hashlib, json, stat, sys, zipfile
# Local validation can override CONFIG before execution. Kaggle uses these private inputs.
CONFIG = globals().get('CONFIG', {
    'bundle': '/kaggle/input/trie-native-arc-submission/native-arc-submission.zip',
    'bundle_receipt': '/kaggle/input/trie-native-arc-submission/bundle-receipt.json',
    'challenges': '/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json',
    'work': '/kaggle/working',
    'run_name': 'native-submission-run-001',
    'solutions': None,
    'worker_timeout_seconds': 30,
    'cohort_budget_seconds': 36000,
})
work = Path(CONFIG['work']).resolve()
bundle_input = Path(CONFIG['bundle']).resolve()
if bundle_input.is_file():
    receipt = json.loads(Path(CONFIG['bundle_receipt']).read_text())
    digest = hashlib.sha256(bundle_input.read_bytes()).hexdigest()
    if digest != receipt['archive_sha256']:
        raise ValueError('Bundle archive hash differs from its receipt')
    extraction = work / 'native-arc-assets'
    if extraction.exists():
        raise FileExistsError('Refusing to reuse an existing bundle extraction directory')
    extraction.mkdir(parents=True)
    with zipfile.ZipFile(bundle_input) as archive:
        for item in archive.infolist():
            target = (extraction / item.filename).resolve()
            if not target.is_relative_to(extraction) or stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError('Unsafe bundle archive member')
        archive.extractall(extraction)
    bundle = extraction / 'native-arc-submission'
else:
    bundle = bundle_input
if not (bundle / 'bundle-manifest.json').is_file():
    raise FileNotFoundError('Hash-pinned bundle manifest is unavailable')
if not Path(CONFIG['challenges']).is_file():
    raise FileNotFoundError('Competition challenges JSON is unavailable')
if not CONFIG['run_name'] or CONFIG['run_name'] in ('.', '..') or Path(CONFIG['run_name']).name != CONFIG['run_name']:
    raise ValueError('run_name must be one plain directory name')
sys.path.insert(0, str(bundle / 'contest'))
from notebook_support import prepare_worker
from arc_submission import run
worker = prepare_worker(bundle, work)
print('Offline matching worker and archive hashes verified.')


In [ ]:
manifest = run(
    CONFIG['challenges'], worker, work / 'submission.json', work / CONFIG['run_name'],
    worker_timeout=CONFIG['worker_timeout_seconds'],
    cohort_budget=CONFIG['cohort_budget_seconds'],
    solutions_path=CONFIG.get('solutions'),
)
print(json.dumps({k: manifest[k] for k in [
    'format_valid', 'tasks', 'queries', 'learned_queries', 'baseline_queries',
    'task_status_counts', 'partner_readiness_threshold'
]}, indent=2))
if 'score' in manifest:
    print(json.dumps({k: v for k, v in manifest['score'].items() if k != 'per_output'}, indent=2))
print('SUBMISSION FILE:', work / 'submission.json')
print('Format validity does not establish hidden-output accuracy.')
